# Paper Figures

This notebook is the single entry point for paper-facing figures.

Current scope:
- dataset overview figure with representative samples from `CRACK500`, `UAV_Crack_Segmentation_Kaggle`, and `PaveCrack1300`
- reproducible export of the selected sample metadata
- B2 supervision-scaling figure built from frozen source-only / promoted B1 / few-shot / upper-bound rows

Selection rule for the main dataset-overview panel:
- do **not** use purely random samples for the main paper figure
- use rank-based crack-area quantiles instead (`Sparse`, `Typical`, `Dense`, `Very dense`)
- keep an optional random panel only as a supplement if needed


In [ ]:
from pathlib import Path
import importlib
import sys

REPO_ROOT = Path.cwd().resolve()
repo_candidates = [REPO_ROOT, *REPO_ROOT.parents]
matched_repo_root = next(
    (candidate for candidate in repo_candidates if (candidate / 'paper_figures.py').exists()),
    None,
)
if matched_repo_root is None:
    raise FileNotFoundError(
        'Could not locate the repository root containing paper_figures.py from the current working directory.'
    )
REPO_ROOT = matched_repo_root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import paper_figures as paper_figures_module
paper_figures_module = importlib.reload(paper_figures_module)

DEFAULT_CATEGORY_SPECS = paper_figures_module.DEFAULT_CATEGORY_SPECS
auto_select_representative_samples = paper_figures_module.auto_select_representative_samples
build_b2_supervision_rows = paper_figures_module.build_b2_supervision_rows
default_dataset_specs = paper_figures_module.default_dataset_specs
load_dataset_records = paper_figures_module.load_dataset_records
render_dataset_overview = paper_figures_module.render_dataset_overview
render_b2_supervision_scaling = paper_figures_module.render_b2_supervision_scaling
select_random_samples = paper_figures_module.select_random_samples
summarize_dataset_records = paper_figures_module.summarize_dataset_records
write_selection_csv = paper_figures_module.write_selection_csv
write_b2_supervision_csv = paper_figures_module.write_b2_supervision_csv

OUTPUT_DIR = REPO_ROOT / 'results' / 'report_assets' / 'paper_figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORY_SPECS = DEFAULT_CATEGORY_SPECS
MANUAL_OVERRIDES = {
    # Example:
    # 'uav_kaggle': {
    #     'Sparse': 'slide1542',
    #     'Typical': 'slide1520',
    # },
}

dataset_specs = default_dataset_specs()
records_by_dataset = load_dataset_records(dataset_specs)
summaries = summarize_dataset_records(records_by_dataset)
summaries


In [ ]:
representative_selections = auto_select_representative_samples(
    records_by_dataset=records_by_dataset,
    category_specs=CATEGORY_SPECS,
    manual_overrides=MANUAL_OVERRIDES,
)
representative_rows = []
for dataset_key, rows in representative_selections.items():
    for row in rows:
        representative_rows.append(
            {
                'dataset': row.dataset_label,
                'category': row.category,
                'sample_id': row.sample_id,
                'fg_ratio_pct': round(row.fg_ratio * 100.0, 2),
                'native_size': f'{row.width}x{row.height}',
                'split_file': row.split_file,
            }
        )
representative_rows


In [ ]:
overview_png = OUTPUT_DIR / 'dataset_overview_representative.png'
overview_csv = OUTPUT_DIR / 'dataset_overview_selected_samples.csv'

render_dataset_overview(
    records_by_dataset=records_by_dataset,
    selections=representative_selections,
    output_path=overview_png,
)
write_selection_csv(representative_selections, overview_csv)

print('Saved representative figure to:', overview_png)
print('Saved representative metadata to:', overview_csv)


In [ ]:
from IPython.display import Image as IPyImage, display

display(IPyImage(filename=str(overview_png)))


In [ ]:
# Optional supplement: a random comparison panel.
# This is useful for sanity checking that the paper figure is representative,
# but it is not recommended as the main qualitative figure.

random_selections = select_random_samples(records_by_dataset, count=4, seed=42)
random_png = OUTPUT_DIR / 'dataset_overview_random_seed42.png'
random_csv = OUTPUT_DIR / 'dataset_overview_random_seed42.csv'

render_dataset_overview(
    records_by_dataset=records_by_dataset,
    selections=random_selections,
    output_path=random_png,
    figure_title='Random dataset samples (supplement only; not the main paper panel)',
)
write_selection_csv(random_selections, random_csv)

print('Saved random comparison figure to:', random_png)
print('Saved random comparison metadata to:', random_csv)


## B2 Supervision Scaling

This Results-facing figure summarizes how performance changes as labeled UAV training data is added after the zero-label `B1` stage.

- x-axis: labeled UAV train samples (`0`, `9`, `19`, `38`)
- `0` on the line corresponds to the promoted `B1` result
- source-only raw transfer is shown as a separate marker at `x = 0`
- dashed lines denote the in-domain upper bound for each backbone


In [ ]:
b2_output_dir = OUTPUT_DIR / 'b2_recovery'
b2_output_dir.mkdir(parents=True, exist_ok=True)
results_csv = REPO_ROOT / 'results' / 'experiments.csv'

b2_rows = build_b2_supervision_rows(results_csv)
b2_rows


In [ ]:
b2_png = b2_output_dir / 'b2_supervision_scaling_holdout.png'
b2_csv = b2_output_dir / 'b2_supervision_scaling_holdout.csv'

render_b2_supervision_scaling(
    rows=b2_rows,
    output_path=b2_png,
    figure_title='UAV hold-out supervision scaling: promoted B1/B2 recovery curves',
)
write_b2_supervision_csv(b2_rows, b2_csv)

print('Saved B2 supervision-scaling figure to:', b2_png)
print('Saved B2 supervision-scaling data to:', b2_csv)


In [ ]:
from IPython.display import Image as IPyImage, display

display(IPyImage(filename=str(b2_png)))
